In [7]:
"""
Open Library Scraper — Book Suggestion Chatbot
Fetches books by subject, saves to CSV and downloads cover images.

Usage:
    python open_library_scraper.py

Requirements:
    pip install requests
"""

import csv
import time
import os
import requests

# ── CONFIG ────────────────────────────────────────────────────────────────────

BASE_URL          = "https://openlibrary.org"
OUTPUT_FILE       = "books_dataset.csv"
BOOKS_PER_SUBJECT = 2000
IMAGES_DIR        = "images"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Subjects: (folder_name, api_url)
SUBJECTS = [
    ("baby books",  f"{BASE_URL}/subjects/infancy.json"),
    ("kittens",     f"{BASE_URL}/subjects/kittens.json"),
    ("cooking",     f"{BASE_URL}/subjects/cooking.json"),
    ("japanese",    f"{BASE_URL}/search.json?q=language%3Ajpn"),
]

CSV_COLUMNS = ["title", "authors", "subject", "publish_year", "description", "rating"]

# ── HELPERS ───────────────────────────────────────────────────────────────────

def get_works_subject(url, limit, offset=0):
    """Fetch works from a /subjects/*.json endpoint with pagination."""
    try:
        r = requests.get(url, params={"limit": limit, "offset": offset}, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return r.json().get("works", [])
    except requests.RequestException as e:
        print(f"  [ERROR] {e}")
        return []


def get_works_search(limit, offset=0):
    """Fetch works from the search endpoint (used for japanese)."""
    try:
        r = requests.get(
            f"{BASE_URL}/search.json",
            params={"q": "language:jpn", "limit": min(limit, 100), "offset": offset,
                    "fields": "key,title,author_name,first_publish_year,cover_i,ratings_average,subject"},
            headers=HEADERS,
            timeout=15,
        )
        r.raise_for_status()
        return r.json().get("docs", [])
    except requests.RequestException as e:
        print(f"  [ERROR] {e}")
        return []


def get_description(work_key):
    """Fetch full description for a work."""
    try:
        r = requests.get(f"{BASE_URL}{work_key}.json", headers=HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
        desc = data.get("description", "")
        if isinstance(desc, dict):
            return desc.get("value", "").strip()
        return (desc or "").strip()
    except requests.RequestException:
        return ""


def get_rating(work_key):
    """Fetch star rating for a work from the ratings endpoint."""
    try:
        r = requests.get(f"{BASE_URL}{work_key}/ratings.json", headers=HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
        avg = data.get("summary", {}).get("average")
        return round(avg, 2) if avg else ""
    except requests.RequestException:
        return ""


def download_image(cover_id, subject_folder, filename):
    """Download highest quality cover image and save to images/<subject>/<filename>.jpg"""
    if not cover_id:
        return
    folder = os.path.join(IMAGES_DIR, subject_folder)
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    if os.path.exists(filepath):
        return  # already downloaded

    # Use -L size for largest available cover
    url = f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg"
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        # Skip placeholder/empty images (Open Library returns tiny gray image if no cover)
        if len(r.content) < 1000:
            return
        with open(filepath, "wb") as f:
            f.write(r.content)
    except requests.RequestException:
        pass


def parse_subject_work(work, subject_name, idx, subject_folder):
    """Parse a work from /subjects/*.json response."""
    cover_id = work.get("cover_id")
    authors  = ", ".join(a.get("name", "Unknown") for a in work.get("authors", []))
    work_key = work.get("key", "")

    description = get_description(work_key) if work_key else ""
    rating      = get_rating(work_key) if work_key else ""
    time.sleep(0.3)

    if cover_id:
        download_image(cover_id, subject_folder, f"book{idx}.jpg")

    return {
        "title":        work.get("title", "Unknown Title"),
        "authors":      authors,
        "subject":      subject_name,
        "publish_year": work.get("first_publish_year", ""),
        "description":  description,
        "rating":       rating,
    }


def parse_search_work(doc, subject_name, idx, subject_folder):
    """Parse a work from /search.json response."""
    cover_id = doc.get("cover_i")
    authors  = ", ".join(doc.get("author_name", []))
    work_key = doc.get("key", "")

    description = get_description(work_key) if work_key else ""
    rating      = get_rating(work_key) if work_key else ""
    time.sleep(0.3)

    if cover_id:
        download_image(cover_id, subject_folder, f"book{idx}.jpg")

    return {
        "title":        doc.get("title", "Unknown Title"),
        "authors":      authors,
        "subject":      subject_name,
        "publish_year": doc.get("first_publish_year", ""),
        "description":  description,
        "rating":       rating,
    }

# ── MAIN ──────────────────────────────────────────────────────────────────────

def scrape():
    print("=" * 55)
    print("  Open Library Scraper — Book Suggestion Chatbot")
    print("=" * 55)

    all_books = []

    for subject_name, api_url in SUBJECTS:
        subject_folder = subject_name
        print(f"\n📚 Scraping: {subject_name}")
        is_search = "search.json" in api_url

        collected = []
        offset    = 0
        batch     = 100  # fetch in batches of 100

        while len(collected) < BOOKS_PER_SUBJECT:
            remaining = BOOKS_PER_SUBJECT - len(collected)
            fetch_n   = min(batch, remaining)

            if is_search:
                works = get_works_search(limit=fetch_n, offset=offset)
            else:
                works = get_works_subject(api_url, limit=fetch_n, offset=offset)

            if not works:
                print(f"   No more works available at offset {offset}")
                break

            print(f"   Batch offset={offset}: {len(works)} works fetched")

            for i, work in enumerate(works):
                global_idx = len(collected) + 1
                if is_search:
                    book = parse_search_work(work, subject_name, global_idx, subject_folder)
                else:
                    book = parse_subject_work(work, subject_name, global_idx, subject_folder)
                collected.append(book)
                print(f"     [{global_idx}/{BOOKS_PER_SUBJECT}] {book['title'][:60]}")

            offset += len(works)
            time.sleep(0.5)

        print(f"   ✅ Collected {len(collected)} books for '{subject_name}'")
        all_books.extend(collected)

    # Write CSV
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(all_books)

    print(f"\n✅ Done! Saved {len(all_books)} rows → {OUTPUT_FILE}")
    print(f"🖼️  Images saved in → {IMAGES_DIR}/")

    # Preview
    if all_books:
        print("\n📖 Sample entry:")
        for col in CSV_COLUMNS:
            val = str(all_books[0][col])
            print(f"   {col}: {val[:80]}")


if __name__ == "__main__":
    scrape()

  Open Library Scraper — Book Suggestion Chatbot

📚 Scraping: baby books
   Batch offset=0: 100 works fetched
     [1/2000] Are You My Mother?
     [2/2000] Owl Babies
     [3/2000] Does a Kangaroo Have a Mother, Too?
     [4/2000] Bug babies
     [5/2000] Portable Pets
     [6/2000] Alligator Baby
     [7/2000] Bird homes
     [8/2000] Progress in Infancy Research
     [9/2000] The reason for a flower
     [10/2000] Happy Baby
     [11/2000] Kitten's first full moon
     [12/2000] The Little Wood Duck
     [13/2000] Annie and the Wild Animals
     [14/2000] Growing Frogs
     [15/2000] Johnny Lion's Book
     [16/2000] I Wonder Why Kangaroos Have Pouches
     [17/2000] Mittens (My First I Can Read)
     [18/2000] チーズスイートホーム [Chīzu suīto hōmu/Chi's Sweet Home] (1)
     [19/2000] Pencil, paper, draw!
     [20/2000] Daisy and the Egg
     [21/2000] Over in the meadow
     [22/2000] Spot goes to the farm
     [23/2000] Never Race a Runaway Pumpkin
     [24/2000] Cowgirl Kate and Cocoa
   

In [8]:
import pandas as pd

In [9]:
df = pd.read_csv("books_dataset.csv")
df.head()

,title,authors,subject,publish_year,description,rating
0,Are You My Mother?,P. D. Eastman,baby books,1960.0,"A baby bird, fallen from his nest, sets out to...",4.28
1,Owl Babies,Martin Waddell,baby books,1975.0,Three owl babies whose mother has gone out in ...,4.00
2,"Does a Kangaroo Have a Mother, Too?",Eric Carle,baby books,1999.0,"Presents the names of animal babies, parents, ...",4.00
3,Bug babies,"Charlotte Guillain, Rebecca Rissman",baby books,2010.0,Descibes in text and photographs how insects a...,NaN
4,Portable Pets,Lorella Rizzati,baby books,1998.0,NaN,NaN


In [10]:
df.shape

(6898, 6)

In [11]:
df.tail()

,title,authors,subject,publish_year,description,rating
6893,Little Bunny's Sleepless Night,Carol Roth,japanese,1999.0,"Little Bunny, an only child, is so lonely that...",NaN
6894,Viaje a la Alcarria,Camilo José Cela,japanese,1953.0,NaN,NaN
6895,The battle for Asia,"Edgar Snow, Edgar Snow",japanese,1941.0,"The intricacies of the far eastern situation, ...",NaN
6896,Nihon kenchiku no katachi,"Nishi, Kazuo",japanese,1983.0,NaN,NaN
6897,The theory of war,P. L. Macdougall,japanese,1856.0,NaN,NaN


In [12]:
df.isna().sum() 

title              0
authors          132
subject            0
publish_year      22
description     3656
rating          5124
dtype: int64

In [13]:
df.duplicated().sum()   

np.int64(25)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6898 entries, 0 to 6897
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         6898 non-null   str    
 1   authors       6766 non-null   str    
 2   subject       6898 non-null   str    
 3   publish_year  6876 non-null   float64
 4   description   3242 non-null   str    
 5   rating        1774 non-null   float64
dtypes: float64(2), str(4)
memory usage: 323.5 KB


In [15]:
df.describe()

,publish_year,rating
count,6876.000000,1774.000000
mean,1980.342350,4.008337
std,57.249626,0.790949
min,1111.000000,1.000000
25%,1977.000000,3.692500
50%,1999.000000,4.000000
75%,2008.000000,4.500000
max,2026.000000,5.000000
